# Paper 2 — Phase 0.5: QCHAN reference-independent spectrum cache

## Why this notebook exists

The frozen Paper 2 specification requires QCHAN to be recomputed from a participant-balanced reference constructed **only from the current training participants**. Therefore, the already-released Paper 1 QCHAN values cannot simply be inserted into nested cross-validation: those values were computed against the Paper 1 frozen cohort reference.

The correct architecture is:

1. use the validated Paper 1 QCHAN v4.0.0 implementation;
2. extract and cache each retained recording's **reference-independent normalized spectrum** once;
3. verify that those spectra reproduce the frozen Paper 1 QCHAN release when the original full-cohort LOSO reference is reconstructed;
4. in Goal 1, rebuild QCHAN references dynamically from the relevant training participants only.

This notebook is **outcome-blind**. Diagnosis, age, and ALSFRS-R are not used to construct spectra or references.

### Pinned validated dependency

`nevena-m3/quality_framework_features`  
commit: `cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8`

No new QCHAN estimator is invented in Paper 2.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

EXPECTED_PAPER1_COMMIT = "cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8"
EXPECTED_RETAINED = 519
EXPECTED_PARTICIPANTS = 224

# If the frozen media paths no longer resolve, set this to the current
# Bamboo_passage_only root, e.g.:
# MEDIA_ROOT_OVERRIDE = Path(r"C:\Users\musikicn\Desktop\Nevena_project\Data_13072026\Bamboo_passage_only")
MEDIA_ROOT_OVERRIDE = None

VERIFY_MEDIA_HASHES = True
REBUILD_CACHE = False

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the Paper 2 repository root. "
        "Run this notebook from the repo or notebooks/ directory."
    )

ROOT = find_project_root()
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
MANIFESTS = ROOT / "data" / "manifests"
EXTERNAL = ROOT / "external" / "quality_framework_features"

CACHE_ROOT = INTERIM / "qchan_v400_spectra"
SPECTRA_DIR = CACHE_ROOT / "spectra"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
SPECTRA_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

print("Paper 2 root:", ROOT)
print("External Paper 1 repo:", EXTERNAL)
print("Spectrum cache:", CACHE_ROOT)


Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
External Paper 1 repo: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\external\quality_framework_features
Spectrum cache: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\data\interim\qchan_v400_spectra


## 1. Verify the pinned Paper 1 implementation

The external repository is kept outside Paper 2's tracked source code and pinned to the exact reviewed commit.

If this cell says the repository is missing, stop and run the PowerShell commands printed by the cell. Do not substitute a newer commit without a deliberate methods/version change.


In [2]:
if not (EXTERNAL / ".git").exists():
    commands = f'''
cd "{ROOT}"
New-Item -ItemType Directory -Force external | Out-Null
git clone https://github.com/nevena-m3/quality_framework_features.git external\\quality_framework_features
git -C external\\quality_framework_features checkout {EXPECTED_PAPER1_COMMIT}
'''
    raise FileNotFoundError(
        "Pinned Paper 1 feature repository is not installed.\n\n"
        "Open PowerShell, activate .venv, then run:\n" + commands
    )

observed_commit = subprocess.check_output(
    ["git", "-C", str(EXTERNAL), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Observed Paper 1 commit:", observed_commit)
assert observed_commit == EXPECTED_PAPER1_COMMIT, (
    "Paper 1 dependency is at the wrong commit.\n"
    f"Expected: {EXPECTED_PAPER1_COMMIT}\n"
    f"Observed: {observed_commit}\n"
    f'Run: git -C "{EXTERNAL}" checkout {EXPECTED_PAPER1_COMMIT}'
)

paper1_src = EXTERNAL / "src"
if str(paper1_src) not in sys.path:
    sys.path.insert(0, str(paper1_src))

from paper1_qc.media import decode_audio_views
from paper1_qc_reviewed.qchan_v400 import (
    ANALYSIS_FEATURES,
    DEFAULT_PARAMETERS,
    analysis_waveform_from_audio_views,
    build_subject_balanced_loso_references,
    compute_reference_relative_features,
    extract_recording_spectrum,
)
from paper1_qc_reviewed.qchan_v400_cohort import (
    canonical_interval_contract,
    intervals_for_recording,
    load_recording_spectrum,
    remove_global_dc,
    resolve_media_path,
    save_recording_spectrum,
)

print("Pinned QCHAN implementation imported successfully.")
print("QCHAN measurement version:", "qchan-v4.0.0")


Observed Paper 1 commit: cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8
Pinned QCHAN implementation imported successfully.
QCHAN measurement version: qchan-v4.0.0


## 2. FFmpeg and frozen-input preflight

Paper 1's canonical media decoder uses FFmpeg/FFprobe. We verify both before touching the cohort.

The governed inputs are:

- the 573-recording Bamboo freeze ledger;
- the 519-recording segmentation decision freeze;
- the frozen strict-speech interval table;
- the frozen 519-recording QCHAN release used **only for reproduction checking**.


In [3]:
ffmpeg = shutil.which("ffmpeg")
ffprobe = shutil.which("ffprobe")

print("ffmpeg:", ffmpeg)
print("ffprobe:", ffprobe)

if not ffmpeg or not ffprobe:
    raise RuntimeError(
        "FFmpeg/FFprobe are required for canonical Paper 1 audio decoding. "
        "Install FFmpeg and make sure both ffmpeg and ffprobe are on PATH, "
        "then restart Jupyter."
    )

ledger_path = RAW / "data" / "bamboo_recording_freeze_ledger.csv"
decisions_path = RAW / "segments" / "frozen_segmentation_decisions.csv"
intervals_path = RAW / "segments" / "frozen_segmentation_intervals.csv"
qchan_release_path = RAW / "features" / "families" / "04_QCHAN" / "features.csv"

for path in [ledger_path, decisions_path, intervals_path, qchan_release_path]:
    assert path.exists(), f"Missing required input: {path}"

ledger = pd.read_csv(ledger_path, low_memory=False)
decisions = pd.read_csv(decisions_path, low_memory=False)
intervals = pd.read_csv(intervals_path, low_memory=False)
qchan_release = pd.read_csv(qchan_release_path, low_memory=False)

eligible_decisions = decisions.loc[
    decisions["segmentation_analysis_eligible"].astype(bool)
].copy()

assert len(eligible_decisions) == EXPECTED_RETAINED
assert eligible_decisions["SubjectID"].nunique() == EXPECTED_PARTICIPANTS
assert len(qchan_release) == EXPECTED_RETAINED

retained = (
    ledger.loc[
        ledger["logical_recording_id"].isin(
            eligible_decisions["logical_recording_id"]
        )
    ]
    .copy()
    .sort_values("logical_recording_id")
    .reset_index(drop=True)
)

assert len(retained) == EXPECTED_RETAINED
assert retained["logical_recording_id"].is_unique

strict_intervals, interval_contract = canonical_interval_contract(
    decisions, intervals
)

print("Retained recordings:", len(retained))
print("Participants:", retained["SubjectID"].nunique())
display(interval_contract)
assert interval_contract["contract_pass"].all()


ffmpeg: C:\ffmpeg\bin\ffmpeg.EXE
ffprobe: C:\ffmpeg\bin\ffprobe.EXE
Retained recordings: 519
Participants: 224


,contract,observed,required,contract_pass
0,canonical_view_profile,strict_speech/primary,strict_speech/primary,True
1,eligible_recordings_have_strict_intervals,519,519,True
2,no_ineligible_recordings_in_strict_table,0,0,True
3,strict_interval_ids_unique,7738,7738,True


## 3. Resolve all frozen media paths

No audio is copied into this repository. The ledger remains the authoritative mapping to source media.

If the original absolute paths have moved, edit only `MEDIA_ROOT_OVERRIDE` in the first configuration cell. Do **not** edit 519 individual ledger rows.


In [4]:
path_audit = []

for row in retained.itertuples(index=False):
    recording_id = str(getattr(row, "logical_recording_id"))
    raw_media_path = getattr(row, "media_path")
    try:
        resolved = resolve_media_path(
            raw_media_path,
            media_root_override=MEDIA_ROOT_OVERRIDE,
        )
        exists = resolved.exists()
        error = ""
    except Exception as exc:
        resolved = None
        exists = False
        error = f"{type(exc).__name__}: {exc}"

    path_audit.append({
        "logical_recording_id": recording_id,
        "raw_media_path": raw_media_path,
        "resolved_media_path": "" if resolved is None else str(resolved),
        "exists": bool(exists),
        "error": error,
    })

path_audit = pd.DataFrame(path_audit)
display(path_audit["exists"].value_counts(dropna=False).rename("n_recordings"))

if not path_audit["exists"].all():
    display(path_audit.loc[~path_audit["exists"]].head(20))
    raise FileNotFoundError(
        f"{(~path_audit['exists']).sum()} retained media paths do not resolve. "
        "Set MEDIA_ROOT_OVERRIDE in the first cell and rerun."
    )

path_audit.to_csv(
    MANIFESTS / "qchan_media_path_audit.csv", index=False
)

print("All retained media paths resolve.")


exists
True    519
Name: n_recordings, dtype: int64

All retained media paths resolve.


## 4. Smoke-test one recording before batch extraction

This verifies:

- the frozen source file is the expected file;
- canonical decoding works;
- strict-speech intervals resolve;
- a QCHAN recording spectrum is measurable.

Only after this passes do we process all 519 recordings.


In [5]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

first = retained.iloc[0]
recording_id = str(first["logical_recording_id"])

resolved_path = Path(
    path_audit.loc[
        path_audit["logical_recording_id"].eq(recording_id),
        "resolved_media_path",
    ].iloc[0]
)

if VERIFY_MEDIA_HASHES:
    observed_sha = sha256_file(resolved_path)
    expected_sha = str(first["media_sha256"])
    assert observed_sha == expected_sha, (
        f"Media SHA mismatch for {recording_id}\n"
        f"Expected: {expected_sha}\nObserved: {observed_sha}"
    )

views = decode_audio_views(
    resolved_path,
    ffmpeg=ffmpeg,
    ffprobe=ffprobe,
    analysis_rate=DEFAULT_PARAMETERS.analysis_sample_rate_hz,
)

waveform_raw = analysis_waveform_from_audio_views(views)
waveform, dc_before, dc_after = remove_global_dc(waveform_raw)
strict, strict_rows = intervals_for_recording(
    strict_intervals, recording_id
)

smoke_spectrum = extract_recording_spectrum(
    waveform,
    DEFAULT_PARAMETERS.analysis_sample_rate_hz,
    strict_speech=strict,
    logical_recording_id=recording_id,
    source_sample_rate_hz=int(views.sample_rate_native),
    parameters=DEFAULT_PARAMETERS,
)

print("Recording:", recording_id)
print("Native sample rate:", int(views.sample_rate_native))
print("Analysis sample rate:", DEFAULT_PARAMETERS.analysis_sample_rate_hz)
print("Strict intervals:", len(strict))
print("DC before:", dc_before)
print("DC after:", dc_after)
print("Spectrum status:", smoke_spectrum.status)
print("Guarded speech support (s):", smoke_spectrum.guarded_speech_support_sec)
print("Valid frames:", smoke_spectrum.valid_frame_count)
print("Spectrum SHA256:", smoke_spectrum.spectrum_sha256)

assert smoke_spectrum.status == "measured"
assert np.isfinite(smoke_spectrum.normalized_psd_per_hz).all()


Recording: C02_1075_1_20260420_240_PSG_BAMBOO
Native sample rate: 44100
Analysis sample rate: 16000
Strict intervals: 10
DC before: -5.668106043270075e-10
DC after: -1.808335095397355e-18
Spectrum status: measured
Guarded speech support (s): 24.792
Valid frames: 2445
Spectrum SHA256: d7f9590bb975933ecc09017637384e9ac7e18a4988964f30d18840e4d3677109


## 5. Build the restart-safe 519-recording spectrum cache

This is the computationally heavier one-time step.

Each recording is decoded and converted into the exact validated QCHAN v4.0.0 reference-independent spectrum. The spectrum cache is stored under `data/interim/`, which is ignored by Git.

If execution is interrupted, rerun the cell: existing valid checkpoints are loaded unless `REBUILD_CACHE=True`.


In [6]:
cache_rows = []
spectra = {}

for idx, row in retained.iterrows():
    recording_id = str(row["logical_recording_id"])
    cache_path = SPECTRA_DIR / f"{recording_id}.npz"

    try:
        if cache_path.exists() and not REBUILD_CACHE:
            spectrum = load_recording_spectrum(cache_path)
            source = "cache"
            media_sha_ok = True
        else:
            resolved_path = Path(
                path_audit.loc[
                    path_audit["logical_recording_id"].eq(recording_id),
                    "resolved_media_path",
                ].iloc[0]
            )

            media_sha_ok = True
            if VERIFY_MEDIA_HASHES:
                media_sha_ok = (
                    sha256_file(resolved_path) == str(row["media_sha256"])
                )
                if not media_sha_ok:
                    raise RuntimeError("Frozen media SHA256 mismatch.")

            views = decode_audio_views(
                resolved_path,
                ffmpeg=ffmpeg,
                ffprobe=ffprobe,
                analysis_rate=DEFAULT_PARAMETERS.analysis_sample_rate_hz,
            )
            waveform, _, _ = remove_global_dc(
                analysis_waveform_from_audio_views(views)
            )
            strict, _ = intervals_for_recording(
                strict_intervals, recording_id
            )

            spectrum = extract_recording_spectrum(
                waveform,
                DEFAULT_PARAMETERS.analysis_sample_rate_hz,
                strict_speech=strict,
                logical_recording_id=recording_id,
                source_sample_rate_hz=int(views.sample_rate_native),
                parameters=DEFAULT_PARAMETERS,
            )

            save_recording_spectrum(spectrum, cache_path)
            source = "computed"

        spectra[recording_id] = spectrum

        cache_rows.append({
            "logical_recording_id": recording_id,
            "SubjectID": str(row["SubjectID"]),
            "status": spectrum.status,
            "support_tier": spectrum.support_tier,
            "guarded_speech_support_sec": spectrum.guarded_speech_support_sec,
            "valid_frame_count": spectrum.valid_frame_count,
            "source_sample_rate_hz": spectrum.source_sample_rate_hz,
            "source_nyquist_hz": spectrum.source_nyquist_hz,
            "source_bandwidth_limited": spectrum.source_bandwidth_limited,
            "spectrum_sha256": spectrum.spectrum_sha256,
            "cache_source": source,
            "media_sha_ok": media_sha_ok,
            "cache_path": str(cache_path.relative_to(ROOT)),
            "error": "",
        })

    except Exception as exc:
        cache_rows.append({
            "logical_recording_id": recording_id,
            "SubjectID": str(row["SubjectID"]),
            "status": "ERROR",
            "support_tier": "",
            "guarded_speech_support_sec": np.nan,
            "valid_frame_count": np.nan,
            "source_sample_rate_hz": np.nan,
            "source_nyquist_hz": np.nan,
            "source_bandwidth_limited": np.nan,
            "spectrum_sha256": "",
            "cache_source": "",
            "media_sha_ok": False,
            "cache_path": str(cache_path.relative_to(ROOT)),
            "error": f"{type(exc).__name__}: {exc}",
        })

    if (idx + 1) % 25 == 0 or (idx + 1) == len(retained):
        print(f"Processed {idx + 1}/{len(retained)}")

cache_index = pd.DataFrame(cache_rows)
cache_index.to_csv(
    MANIFESTS / "qchan_spectrum_cache_index.csv", index=False
)

display(cache_index["status"].value_counts(dropna=False).rename("n_recordings"))

errors = cache_index.loc[cache_index["status"].eq("ERROR")]
if len(errors):
    display(errors[["logical_recording_id", "error"]].head(20))
    raise RuntimeError(
        f"{len(errors)} QCHAN spectrum extractions failed. "
        "Resolve failures before proceeding."
    )

assert len(spectra) == EXPECTED_RETAINED
assert cache_index["logical_recording_id"].nunique() == EXPECTED_RETAINED
assert cache_index["status"].eq("measured").all()
assert cache_index["spectrum_sha256"].ne("").all()

print("REFERENCE-INDEPENDENT QCHAN SPECTRUM CACHE: PASS")


Processed 25/519
Processed 50/519
Processed 75/519
Processed 100/519
Processed 125/519
Processed 150/519
Processed 175/519
Processed 200/519
Processed 225/519
Processed 250/519
Processed 275/519
Processed 300/519
Processed 325/519
Processed 350/519
Processed 375/519
Processed 400/519
Processed 425/519
Processed 450/519
Processed 475/519
Processed 500/519
Processed 519/519


status
measured    519
Name: n_recordings, dtype: int64

REFERENCE-INDEPENDENT QCHAN SPECTRUM CACHE: PASS


## 6. Reproduce the frozen Paper 1 QCHAN release

This is a critical equivalence test.

We temporarily reconstruct the **original Paper 1 full-cohort subject-balanced LOSO reference** and recompute the four released QCHAN values. These reproduced values are compared with the frozen release.

This full-cohort reference is used **only for validation of the cache**. It will not be used for Paper 2 clinical inference.


In [7]:
reference_metadata = retained[
    ["logical_recording_id", "SubjectID"]
].copy()

reference_metadata = reference_metadata.rename(
    columns={"SubjectID": "subject_id"}
)
reference_metadata["logical_recording_id"] = (
    reference_metadata["logical_recording_id"].astype(str)
)
reference_metadata["subject_id"] = (
    reference_metadata["subject_id"].astype(str)
)
reference_metadata["task_stratum"] = "BAMBOO_PASSAGE"

references = build_subject_balanced_loso_references(
    spectra,
    reference_metadata,
    parameters=DEFAULT_PARAMETERS,
)

assert len(references) == EXPECTED_RETAINED

recomputed_rows = []
for recording_id in reference_metadata["logical_recording_id"]:
    features_row = compute_reference_relative_features(
        spectra[recording_id],
        references[recording_id],
        parameters=DEFAULT_PARAMETERS,
    )
    recomputed_rows.append(features_row)

qchan_recomputed = pd.DataFrame(recomputed_rows)

comparison = qchan_release[
    ["logical_recording_id", *ANALYSIS_FEATURES]
].merge(
    qchan_recomputed[
        ["logical_recording_id", *ANALYSIS_FEATURES]
    ],
    on="logical_recording_id",
    suffixes=("_frozen", "_recomputed"),
    how="inner",
    validate="one_to_one",
)

assert len(comparison) == EXPECTED_RETAINED

check_rows = []
for feature in ANALYSIS_FEATURES:
    frozen = pd.to_numeric(
        comparison[f"{feature}_frozen"], errors="coerce"
    ).to_numpy(float)
    recomputed = pd.to_numeric(
        comparison[f"{feature}_recomputed"], errors="coerce"
    ).to_numpy(float)

    diff = recomputed - frozen
    finite = np.isfinite(frozen) & np.isfinite(recomputed)

    max_abs = float(np.max(np.abs(diff[finite]))) if finite.any() else np.nan
    mean_abs = float(np.mean(np.abs(diff[finite]))) if finite.any() else np.nan
    equivalent = bool(
        np.allclose(
            frozen,
            recomputed,
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )
    )

    check_rows.append({
        "feature": feature,
        "n_compared": int(finite.sum()),
        "max_abs_difference": max_abs,
        "mean_abs_difference": mean_abs,
        "equivalent_rtol1e-9_atol1e-9": equivalent,
    })

reproduction_check = pd.DataFrame(check_rows)
display(reproduction_check)

reproduction_check.to_csv(
    MANIFESTS / "qchan_paper1_reproduction_check.csv",
    index=False,
)

if not reproduction_check["equivalent_rtol1e-9_atol1e-9"].all():
    raise AssertionError(
        "Cached spectra do not reproduce the frozen Paper 1 QCHAN release "
        "to the prespecified numerical tolerance. Do not proceed to Goal 1."
    )

print("PAPER 1 QCHAN NUMERICAL REPRODUCTION: PASS")


,feature,n_compared,max_abs_difference,mean_abs_difference,equivalent_rtol1e-9_atol1e-9
0,qchan_ltas_distance_db,519,1.776357e-15,8.214367e-17,True
1,qchan_rolloff95_deficit_hz,519,2.273737e-13,5.288106e-15,True
2,qchan_highband_ratio_deficit,519,9.887924e-17,2.595479e-17,True
3,qchan_tilt_steepening_db_per_oct,519,6.661338e-16,2.479752e-17,True


PAPER 1 QCHAN NUMERICAL REPRODUCTION: PASS


## 7. Write the Phase 0.5 readiness manifest

Goal 1 may use QCHAN only after this manifest reports PASS.

The next notebook will never load the frozen full-cohort QCHAN values as predictors. It will load the reference-independent spectrum cache and rebuild QCHAN inside the appropriate inner/outer training partitions.


In [8]:
ready = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "paper1_repo_commit": observed_commit,
    "qchan_measurement_version": "qchan-v4.0.0",
    "retained_recordings": int(len(cache_index)),
    "participants": int(retained["SubjectID"].nunique()),
    "all_media_paths_resolved": bool(path_audit["exists"].all()),
    "all_media_hashes_verified": bool(
        cache_index["media_sha_ok"].all()
    ) if VERIFY_MEDIA_HASHES else None,
    "all_spectra_measured": bool(
        cache_index["status"].eq("measured").all()
    ),
    "paper1_release_reproduced": bool(
        reproduction_check["equivalent_rtol1e-9_atol1e-9"].all()
    ),
    "paper2_qchan_policy": (
        "Reference-independent spectra cached once; "
        "QCHAN references must be rebuilt from training participants only "
        "inside nested CV."
    ),
}

ready_path = MANIFESTS / "qchan_cache_ready.json"
ready_path.write_text(
    json.dumps(ready, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([ready]).T.rename(columns={0: "value"}))

assert ready["all_media_paths_resolved"]
assert ready["all_spectra_measured"]
assert ready["paper1_release_reproduced"]

print("\nPHASE 0.5 QCHAN CACHE: READY FOR GOAL 1")
print("Manifest:", ready_path.relative_to(ROOT))


,value
created_utc,2026-08-26T17:02:05.724823+00:00
paper1_repo_commit,cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8
qchan_measurement_version,qchan-v4.0.0
retained_recordings,519
participants,224
all_media_paths_resolved,True
all_media_hashes_verified,True
all_spectra_measured,True
paper1_release_reproduced,True
paper2_qchan_policy,Reference-independent spectra cached once; QCH...



PHASE 0.5 QCHAN CACHE: READY FOR GOAL 1
Manifest: data\manifests\qchan_cache_ready.json
